# Independently verified dashboard release candidate

## Purpose
Build a sealed local candidate without changing the approved release or analytical evidence.

## Hypothesis
Candidate construction and independent validation can complete while the existing approved pointer remains byte-identical. An invalid candidate must not be approved.

## Methodology
Reuse hash-bound, previously approved analytical evidence; do not rerun models. Build disjoint facts and dimension contracts, then validate the on-disk candidate in a separate project-venv process. Promotion is a separate explicit call after verification and requires a fresh verification under an exclusive promotion lock. A single atomic pointer replacement is the approval commit point. Historical evidence and source files are immutable.

## Evidence
The code displays the candidate identity, builder validation count and independent validation receipt. Seconds remain canonical; differences of rates use percentage points. Fixed effects and quantiles cannot be recombined.


In [1]:
from pathlib import Path
import sys, json
ROOT=Path.cwd().resolve()
if not (ROOT/'data/cleaned/swift_payments_cleaned.csv').exists(): ROOT=ROOT.parent
assert Path(sys.executable).resolve()==(ROOT/'.venv/Scripts/python.exe').resolve()
sys.path.insert(0,str(ROOT/'src'))
import pandas as pd
pd.set_option('display.max_columns',30)

from phase2_release import build_dashboard_release

In [2]:
report=build_dashboard_release(ROOT)
print(json.dumps({'RUN_ID':report['RUN_ID'],'RELEASE_STATE':report['RELEASE_STATE'],'BUILDER_CHECKS':report['VALIDATION_CHECKS_PASSED']},indent=2))

{
  "RUN_ID": "20261115T021543009685Z",
  "RELEASE_STATE": "CANDIDATE",
  "BUILDER_CHECKS": 234
}


In [3]:
from release_promotion import verify_candidate
release=ROOT/'dashboard/data/releases'/report['RUN_ID']
receipt,verification=verify_candidate(ROOT,release)
print(json.dumps({k:v for k,v in verification.items() if k!='CHECKS'},indent=2))
display(pd.read_csv(release/'release_validation.csv'))
display(pd.read_csv(release/'dashboard_table_contract.csv'))

{
  "STATUS": "PASSED",
  "RUN_ID": "20261115T021543009685Z",
  "RELEASE_PATH": "dashboard/data/releases/20261115T021543009685Z",
  "MANIFEST_SHA256": "3ee59074262483881a561c307691c6d4f0d83bff40c248fb6eb8e527634494ee",
  "VERIFIER_SHA256": "ba38ea9fe7ecda0ae330752ee313a0496bc475742950feb37121d2a87ebc3ef5",
  "CHECKS_PASSED": 600,
  "VERIFIED_AT": "2026-11-15T02:15:59.710197+00:00",
  "TECHNICAL_CHART_QA": {
    "CONTACT_SHEET_GENERATED": true,
    "IMAGE_OPEN_CHECK": "PASSED",
    "CHART_HASH_CHECK": "PASSED",
    "VISUAL_REVIEW_STATUS": "PENDING_MANUAL_OR_AGENT_REVIEW",
    "DIMENSIONS": {
      "duration_nested_model_performance.png": [
        1492,
        753
      ],
      "F01_evidence.png": [
        1492,
        753
      ],
      "F02_evidence.png": [
        1492,
        753
      ],
      "F03_evidence.png": [
        1492,
        753
      ],
      "F04_evidence.png": [
        1492,
        753
      ],
      "F05_evidence.png": [
        1492,
        753
      ],
   

,CHECK,PASSED
0,Source population,True
1,Source row alignment,True
2,Canonical rule schema and unique IDs,True
3,Canonical statuses,True
4,Phase1 conclusions preserved,True
...,...,...
229,release_validation no direct identifiers,True
230,release_validation finite numeric values,True
231,release_table_inventory valid schema,True
232,release_table_inventory no direct identifiers,True


,TABLE_NAME,GRAIN,PRIMARY_KEY_OR_GRAIN,AVAILABLE_DIMENSIONS,MEASURES,DENOMINATOR,ADDITIVE_MEASURES,NON_ADDITIVE_MEASURES,ALLOWED_SLICERS,DISALLOWED_SLICERS,SOURCE_NOTEBOOK,SOURCE_TABLE,LIMITATIONS,UNIT_NOTE,EMPTY_SELECTION,FIXED_ESTIMAND_FILTERING
0,FactPortfolioOverview,PORTFOLIO_KEY,PORTFOLIO_KEY,NONE,N;AMOUNT_SUM_USD;DURATION_SUM_SECONDS;MEDIAN_D...,Global source n=1048575; no segment filters su...,N;AMOUNT_SUM_USD;DURATION_SUM_SECONDS;LABELED_...,MEDIAN_DURATION_SECONDS;P95_DURATION_SECONDS;P...,NONE,Routing;Currency;Party;Threshold;Participant;F...,09_dashboard_data_prep.ipynb,Canonical cleaned CSV aggregate,Global source n=1048575; no segment filters s...,Transactions / USD / seconds,"BLANK / unavailable, not a fabricated zero",Selecting evidence rows never refits models or...
1,FactRouteProfile,ROUTING_TYPE,ROUTING_TYPE,Routing,N;AMOUNT_SUM_USD;DURATION_SUM_SECONDS;MEDIAN_D...,Each route N; disjoint routes; exact-grain qua...,N;AMOUNT_SUM_USD;DURATION_SUM_SECONDS;LABELED_N,MEDIAN_DURATION_SECONDS;P90_DURATION_SECONDS;P...,Routing,Currency;Party;Threshold;Participant;FX;Rule;D...,09_dashboard_data_prep.ipynb,Canonical cleaned CSV route aggregate,Each route N; disjoint routes; exact-grain qu...,Transactions / USD / seconds,"BLANK / unavailable, not a fabricated zero",Selecting evidence rows never refits models or...
2,FactCurrencyProfile,CURRENCY,CURRENCY,Currency,N;MEAN_USD_AMOUNT;MEDIAN_USD_AMOUNT;P25_USD_AM...,Per-category N; SHARE_OF_ALL uses global104857...,N;AMOUNT_SUM_USD;DURATION_SUM_SECONDS;LABELED_...,MEAN_USD_AMOUNT;MEDIAN_USD_AMOUNT;P25_USD_AMOU...,Currency,Routing;Party;Threshold;Participant;FX;Rule;Da...,10_final_analytical_integration.ipynb,outputs/tables/final_evidence/currency_final_p...,Sparse category flag retained. Sum numerator/d...,USD amounts; duration seconds; rates/shares fr...,"BLANK / unavailable, not a fabricated zero",Selecting evidence rows never refits models or...
3,FactPartyProfile,PARTY_PAIR,PARTY_PAIR,Party,N;MEAN_USD_AMOUNT;MEDIAN_USD_AMOUNT;P25_USD_AM...,Per-category N; SHARE_OF_ALL uses global104857...,N;AMOUNT_SUM_USD;DURATION_SUM_SECONDS;LABELED_...,MEAN_USD_AMOUNT;MEDIAN_USD_AMOUNT;P25_USD_AMOU...,Party,Routing;Currency;Threshold;Participant;FX;Rule...,10_final_analytical_integration.ipynb,outputs/tables/final_evidence/party_final_prof...,Sparse category flag retained. Sum numerator/d...,USD amounts; duration seconds; rates/shares fr...,"BLANK / unavailable, not a fabricated zero",Selecting evidence rows never refits models or...
4,FactCurrencyPartyComposition,CURRENCY;PARTY_PAIR,CURRENCY;PARTY_PAIR,Currency;Party,N;PARTY_DENOMINATOR_N;P_CURRENCY_GIVEN_PARTY;C...,P(currency|party) denominator is all currency ...,N,PARTY_DENOMINATOR_N;P_CURRENCY_GIVEN_PARTY;CUR...,Currency;Party,Routing;Threshold;Participant;FX;Rule;Date;Eff...,10_final_analytical_integration.ipynb,currency_given_party.csv; party_given_currency...,Stored conditional denominators repeat: NEVER ...,Transactions; conditional probabilities fraction,"BLANK / unavailable, not a fabricated zero",Selecting evidence rows never refits models or...
5,FactRouteCurrencyParty,ROUTING_TYPE;CURRENCY;PARTY_PAIR,ROUTING_TYPE;CURRENCY;PARTY_PAIR,Routing;Currency;Party,VISIBLE_N;AMOUNT_SUM_USD;DURATION_SUM_SECONDS;...,Only visible cells n>=20; denominator VISIBLE_...,VISIBLE_N;AMOUNT_SUM_USD;DURATION_SUM_SECONDS;...,MEDIAN_DURATION_SECONDS;P95_DURATION_SECONDS;M...,Routing;Currency;Party,Threshold;Participant;FX;Rule;Date;Effect;Endp...,09_dashboard_data_prep.ipynb,Canonical cleaned CSV route/currency/party agg...,4365 suppressed rows are NOT in this fact and ...,Transactions / USD / seconds,"BLANK / unavailable, not a fabricated zero",Selecting evidence rows never refits models or...
6,FactSuppressionCoverage,COVERAGE_KEY,COVERAGE_KEY,NONE,TOTAL_SOURCE_N;VISIBLE_CUBE_N;SUPPRESSED_N;GLO...,Global coverage only; cannot filter by route/c...,NaN,TOTAL_SOURCE_N;VISIBLE_CUBE_N;SUPPRESSED_N;GLO...,NONE,Routing;Currency;Party;Threshold;Partic

## Findings and limitations

Calculated results and exact numerical conclusions are recorded in the displayed evidence and generated final report, not preset in this notebook. F07/F08 remain rejected universal hypotheses. F11 remains insufficient for a portfolio-wide bank-independent currency effect; F12 remains insufficient for a Direct-delay mechanism. Page 7 is retrospective supporting duplicate/token QA only. No external validation or causal effect is claimed. Quantiles and fixed estimates are non-additive. The 4365 suppressed cube rows have only global coverage; filtered full-population coverage is unavailable.

## Business implications
Compare rate and workload separately, monitor observability and conditional duration associations, and use structural checks as data-quality controls. No savings, SLA improvements, fraud accuracy or token predictive novelty is invented.

## Completion summary
The executed output contains population, validation status and output locations. Detailed file inventory, limitations, all eight page decisions and remaining implementation requirements are in docs/phase2_final_integration_report.md. The immutable release inventory and manifest list every generated input with content hashes. Future Power BI work must enforce allowed slicers and guarded measures; it is not performed here.


## Interpretation
PASSED means the sealed candidate passed builder and independent on-disk validation. This notebook never promotes a candidate automatically. Use release_promotion.promote_release only after review; it independently verifies again before atomic approval. The current_release.json pointer and its manifest/receipt hashes are the only approval authority. The historical top-level contract is a compatibility artifact; resolve the release-specific CONTRACT from the approved pointer.

## Release limitations
Atomicity concerns a single local approval pointer. Filesystem/hardware durability and malicious concurrent payload mutation are outside this local cooperative workflow guarantee. Promotion locks serialize cooperating promoters and stale candidates fail. No Power BI has been constructed.


Automated chart QA verifies PNG integrity and hashes only; VISUAL_REVIEW_STATUS remains PENDING_MANUAL_OR_AGENT_REVIEW. Explicit nested-model references, cross-phase sample identity, dynamic rule counts and canonical endpoint provenance are release gates. The legacy verifier is blocked.
